# Beyond - Speculative Decoding

This notebook proves greedy speculative decoding is lossless on the StoryLM ladder, sweeps the acceptance curve across block sizes and drafters, runs an honest wall-clock benchmark, trains a multi-token-prediction head on frozen StoryLM-5M, and closes with self-speculation and the cost model.

1. Read the lesson page (`docs/beyond/specdec.md`).
2. Open this notebook with `./notebook.sh specdec`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
import time

import torch
import matplotlib.pyplot as plt

from g2c.artifacts import load_model_artifact_with_tokenizer
from g2c.sampling import generate
from g2c.specdec import SpecStats, speculative_generate

device = "mps" if torch.backends.mps.is_available() else "cpu"

MODELS = {}
try:
    for name in ("StoryLM-1M-base", "StoryLM-5M-base",
                 "StoryLM-30M-base"):
        MODELS[name] = load_model_artifact_with_tokenizer(
            name, device=device
        )
except FileNotFoundError as exc:
    raise RuntimeError(
        "StoryLM reference checkpoints are missing - run "
        "./checkpoints.sh first."
    ) from exc

target = MODELS["StoryLM-30M-base"].model
tokenizer = MODELS["StoryLM-30M-base"].tokenizer
VOCAB = target.vocab_size


def encode(text):
    return torch.tensor(
        tokenizer.encode_with_vocab_size(text, VOCAB),
        dtype=torch.long,
    )


print(f"device: {device}  target vocab: {VOCAB}")

## Exercise 1 — Prove losslessness on the real ladder

StoryLM-1M drafts for StoryLM-30M. Greedy speculative output must be byte-identical to plain greedy decoding — the drafter can change the pass count, never the tokens.

In [ ]:
drafter = MODELS["StoryLM-1M-base"].model

PROMPTS = [
    "Once upon a time",
    "The little dog",
    "One day, Lily and her mom",
]

for prompt in PROMPTS:
    ids = encode(prompt)
    plain = generate(target, ids, 60, temperature=0.0)
    spec, stats = speculative_generate(target, drafter, ids, 60, k=4)
    assert spec.tolist() == plain.tolist(), "speculative output diverged!"
    print(f"{prompt!r}: identical.  {stats}")

print()
print(tokenizer.decode(spec.tolist()))

In [ ]:
"Question: The assertion passed: speculative and plain greedy outputs were identical on every prompt. Explain WHY this is guaranteed rather than lucky - what does greedy verification accept, and what token is emitted at the first disagreement? Then read your SpecStats: what does tokens-per-pass above 1.0 mean in terms of serial target passes?"
"Answer: "

## Exercise 2 — Sweep the acceptance curve

Tokens-per-pass across block sizes for both drafters. Each extra draft position only pays off if the whole prefix before it survives.

In [ ]:
import itertools

drafters = {"1M": MODELS["StoryLM-1M-base"].model,
            "5M": MODELS["StoryLM-5M-base"].model}
KS = (1, 2, 4, 8)

ids = encode("Once upon a time")
results = {}
for (dname, dmodel), k in itertools.product(drafters.items(), KS):
    _, stats = speculative_generate(target, dmodel, ids, 80, k=k)
    results[(dname, k)] = stats
    print(f"drafter {dname}  k={k}:  {stats}")

plt.figure(figsize=(7, 4))
for dname in drafters:
    plt.plot(KS, [results[(dname, k)].tokens_per_pass for k in KS],
             marker="o", label=f"{dname} drafter")
plt.axhline(1.0, color="gray", lw=0.8, label="plain decoding")
plt.xlabel("draft block size k")
plt.ylabel("tokens per target pass")
plt.title("Acceptance sweep")
plt.legend()
plt.show()

In [ ]:
"Question: Read your sweep: where does a bigger k stop paying, and why does the curve flatten (think about what has to happen for draft position k to matter at all)? Comparing the 1M and 5M drafters: how much acceptance does the bigger drafter buy, and is it worth its roughly 4-5x higher drafting cost on this evidence?"
"Answer: "

## Exercise 3 — The wall-clock honesty benchmark

Tokens-per-pass is accounting; tokens-per-second is engineering. Measure both. At this scale each verification pass recomputes the full prefix and the drafter loops in Python, so "the accounting improved and the clock didn't" is a legitimate result.

In [ ]:
def sync():
    if device == "mps":
        torch.mps.synchronize()


def timed(fn, repeats=3):
    best = float("inf")
    out = None
    for _ in range(repeats):
        sync()
        start = time.perf_counter()
        out = fn()
        sync()
        best = min(best, time.perf_counter() - start)
    return out, best


ids = encode("Once upon a time")
N = 80
BEST_K = 4  # set from your Exercise 2 curve

_, t_plain = timed(lambda: generate(target, ids, N, temperature=0.0))
(_, stats), t_spec = timed(
    lambda: speculative_generate(target, drafter, ids, N, k=BEST_K)
)
print(f"plain:       {N / t_plain:6.1f} tok/s   ({t_plain:.2f}s)")
print(f"speculative: {N / t_spec:6.1f} tok/s   ({t_spec:.2f}s)   "
      f"tokens/pass {stats.tokens_per_pass:.2f}")

In [ ]:
"Question: Report tokens-per-pass next to tokens-per-second. If they tell different stories, name the two costs in this implementation that the pass count ignores. What machinery would production serving add before the pass savings became latency savings?"
"Answer: "

## Exercise 4 — Train the MTP head

Bolt `MTPHead` onto frozen StoryLM-5M and train it on TinyStories to predict two tokens ahead. The base never moves — the head's `parameters()` simply excludes it, Module 13B's freeze-by-omission trick.

In [ ]:
from g2c.artifacts import load_tokenized_corpus_artifact
from g2c.pretraining import get_lm_batch
from g2c.specdec import MTPHead, hidden_states, mtp_loss
from g2c.training import AdamW

CORPUS_NAME = "StoryLM-tinystories-100MB-v4096"
try:
    corpus = load_tokenized_corpus_artifact(CORPUS_NAME)
except FileNotFoundError as exc:
    raise RuntimeError(
        "TinyStories artifacts are missing - run "
        "./datasets.sh --tiny first."
    ) from exc
pair = corpus.split(train_fraction=0.9, chunk_tokens=100_000, seed=12)

base = MODELS["StoryLM-5M-base"].model
head = MTPHead(base).to(device)
n_head = sum(p.numel() for p in head.parameters())
n_base = sum(p.numel() for p in base.parameters())
print(f"trainable head: {n_head:,} params  "
      f"({n_head / n_base:.1%} of the frozen base)")

In [ ]:
opt = AdamW(head.parameters(), lr=1e-3)
gen = torch.Generator().manual_seed(0)

STEPS, B, T = 400, 16, 128
losses = []
for step in range(1, STEPS + 1):
    x, _ = get_lm_batch(pair.train, B, T, generator=gen)
    x = x.to(device)
    h = hidden_states(base, x)
    logits2 = head(h[:, :-1], x[:, 1:])
    loss = mtp_loss(logits2, x)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 50 == 0:
        print(f"step {step}: mtp loss {loss.item():.3f}")

plt.figure(figsize=(7, 4))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("two-ahead cross-entropy")
plt.title("MTP head training (base frozen)")
plt.show()

In [ ]:
with torch.no_grad():
    x, _ = get_lm_batch(pair.val, 32, 128, generator=gen)
    x = x.to(device)
    h = hidden_states(base, x)
    logits1 = h @ base.token_embed.weight.T + base.head_bias
    one_ahead = (logits1[:, :-1].argmax(-1) == x[:, 1:]).float().mean()
    logits2 = head(h[:, :-1], x[:, 1:])
    two_ahead = (logits2[:, :-1].argmax(-1) == x[:, 2:]).float().mean()
print(f"one-ahead top-1 (base):    {one_ahead:.3f}")
print(f"two-ahead top-1 (MTP head): {two_ahead:.3f}")

## Exercise 5 — Self-speculation

Decode with the head as the drafter: each iteration takes the base's own next token for free, drafts one more with the head, and verifies. One honesty note before you read the stats: this simple loop pays a separate proposal pass per iteration, while production MTP folds proposal into the *previous* verification pass — so count `target_passes` as verification passes and carry the caveat into your answer.

In [ ]:
from g2c.specdec import greedy_verify, mtp_propose


def self_speculative_generate(base, head, prompt_ids, max_new_tokens):
    full = prompt_ids.detach().cpu().clone()
    stats = SpecStats()
    while stats.generated < max_new_tokens:
        next1, next2 = mtp_propose(base, head, full)
        draft = torch.tensor([next2])
        ctx = full[-(base.max_seq_len - 2):]
        seq = torch.cat([ctx, torch.tensor([next1]), draft])
        logits = base(seq.to(base.device).unsqueeze(0))
        block = logits[0, -2:, :].cpu()
        n_acc, nxt = greedy_verify(draft, block)
        new_ids = torch.cat(
            [torch.tensor([next1]), draft[:n_acc], torch.tensor([nxt])]
        )
        new_ids = new_ids[: max_new_tokens - stats.generated]
        stats.record(drafted=1, accepted=n_acc,
                     generated=int(new_ids.numel()))
        full = torch.cat([full, new_ids])
    return full, stats


ids = encode("Once upon a time")
plain = generate(base, ids, 60, temperature=0.0)
self_spec, stats = self_speculative_generate(base, head, ids, 60)
print(f"identical to plain greedy: {self_spec.tolist() == plain.tolist()}")
print(stats)
print(f"MTP draft acceptance: {stats.acceptance_rate:.1%}")

In [ ]:
"Question: Two readings. First: how much worse is the head at two-ahead than the base is at one-ahead, and why is predicting x_{t+2} strictly harder (what must the head marginalize over)? Second: compare self-speculation's acceptance and tokens-per-pass against your Exercise 2 separate-drafter numbers, remembering the proposal-pass caveat - what is the structural trade between a built-in drafter and a separate one?"
"Answer: "

## Exercise 6 — Written: the cost model

In [ ]:
"Question: With c = cost(drafter pass) / cost(target pass), one speculative iteration costs 1 + k*c target-units and yields E[accepted] + 1 tokens. Derive tokens per unit of target-compute for your best Exercise 2 configuration (estimate c from the parameter ratio and state the estimate), find the break-even acceptance for k = 4, and interpret GLM-5's reported 2.76-token acceptance length in these terms given that its MTP drafter rides the trunk pass nearly free."
"Answer: "

When complete, ask a coding agent to grade your notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.